<a href="https://colab.research.google.com/github/shivangi5678/Sales-KPI-Automation-Pipeline/blob/main/AUTOMATION.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install pandas openpyxl requests python-dotenv

In [2]:
import pandas as pd
import requests

# 1. Public API se sample transactional/product data fetch karna
url = "https://fakestoreapi.com/products"
response = requests.get(url)
data = response.json()

# 2. DataFrame create karna
df = pd.DataFrame(data)

# 3. Data inspection & transformation
df['rating_count'] = df['rating'].apply(lambda x: x['count'] if isinstance(x, dict) else 0)
df['rating_rate'] = df['rating'].apply(lambda x: x['rate'] if isinstance(x, dict) else 0)
df.drop(columns=['rating', 'image', 'description'], inplace=True)

# 4. Estimated Sales Metrics calculate karna
df['estimated_revenue'] = (df['price'] * df['rating_count']).round(2)

print("Data Extracted Successfully! Total Records:", len(df))
df.head()

Data Extracted Successfully! Total Records: 20


,id,title,price,category,rating_count,rating_rate,estimated_revenue
0,1,"Fjallraven - Foldsack No. 1 Backpack, Fits 15 ...",109.95,men's clothing,120,3.9,13194.0
1,2,Mens Casual Premium Slim Fit T-Shirts,22.30,men's clothing,259,4.1,5775.7
2,3,Mens Cotton Jacket,55.99,men's clothing,500,4.7,27995.0
3,4,Mens Casual Slim Fit,15.99,men's clothing,430,2.1,6875.7
4,5,John Hardy Women's Legends Naga Gold & Silver ...,695.00,jewelery,400,4.6,278000.0


In [3]:
from datetime import datetime

# 1. High-level business KPIs calculate karna
total_products = len(df)
total_est_revenue = df['estimated_revenue'].sum()
avg_product_price = df['price'].mean()
avg_rating = df['rating_rate'].mean()

# 2. Category-wise Performance Summary
category_summary = df.groupby('category').agg(
    product_count=('id', 'count'),
    avg_price=('price', 'mean'),
    total_revenue=('estimated_revenue', 'sum'),
    avg_rating=('rating_rate', 'mean')
).reset_index()

category_summary['avg_price'] = category_summary['avg_price'].round(2)
category_summary['total_revenue'] = category_summary['total_revenue'].round(2)
category_summary['avg_rating'] = category_summary['avg_rating'].round(2)

# 3. Top 3 Revenue Driving Products
top_performers = df.sort_values(by='estimated_revenue', ascending=False)[['title', 'category', 'price', 'estimated_revenue']].head(3)

# 4. Multi-sheet Excel Report generate karna
today_str = datetime.today().strftime('%Y-%m-%d')
report_filename = f"daily_sales_kpi_report_{today_str}.xlsx"

with pd.ExcelWriter(report_filename, engine='openpyxl') as writer:
    df.to_excel(writer, sheet_name='Raw Data', index=False)
    category_summary.to_excel(writer, sheet_name='Category Summary', index=False)
    top_performers.to_excel(writer, sheet_name='Top Performers', index=False)

print(f"Report Successfully Generated: {report_filename}")
print(f"Total Revenue: ${total_est_revenue:,.2f} | Total Products: {total_products}")
category_summary

Report Successfully Generated: daily_sales_kpi_report_2026-09-05.xlsx
Total Revenue: $838,090.61 | Total Products: 20


,category,product_count,avg_price,total_revenue,avg_rating
0,electronics,6,332.50,434341.60,3.48
1,jewelery,4,221.00,294855.00,3.35
2,men's clothing,4,51.06,53840.40,3.70
3,women's clothing,6,26.29,55053.61,3.68


In [5]:
from google.colab import files

print("========================================")
print("📊 AUTOMATION PIPELINE EXECUTED")
print(f"Total Products Processed: {total_products}")
print(f"Total Estimated Revenue: ${total_est_revenue:,.2f}")
print("========================================")

# Excel report direct local system me download ho jayegi
files.download(report_filename)
print(f"Report downloaded: {report_filename}")

📊 AUTOMATION PIPELINE EXECUTED
Total Products Processed: 20
Total Estimated Revenue: $838,090.61


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Report downloaded: daily_sales_kpi_report_2026-09-05.xlsx
